# Tiempo real: un consumidor del directo

Este notebook recorre el hito 6 de EDsim desde el otro lado de Kafka: el del **consumidor**.
Lanza un directo corto (`edsim directo lanzar`), lo lee a la vez con el consumidor del
paquete (`edsim.live.kafka.read_run`) y mantiene el **censo en vivo**: quién está en
urgencias y dónde, minuto a minuto. Después comprueba lo que el hito promete: que lo leído en
vivo es lo mismo que el batch, aunque Kafka reparta los eventos en particiones, los entregue
«al menos una vez» y algunos lleguen repetidos.

Toda la lógica vive en el paquete y está testeada (`edsim.live.tracker` y `edsim.live.kafka`);
aquí solo se llama a sus funciones, se dibuja y se explica (decisión 18).

## 1. Cómo usar este notebook

**Con Kafka** (lo normal en local). Hacen falta el broker y el cliente de Kafka:

1. En una terminal, en la raíz del proyecto: `docker compose up --wait` y
   `uv run edsim directo preparar` (crea los cuatro topics; la segunda vez dice «Ya estaban»).
2. `uv sync --all-packages --extra kafka`: instala `confluent-kafka` en `.venv`. Ojo: un
   `uv sync` sin `--extra kafka` lo vuelve a quitar.
3. Abre el notebook en Cursor con el kernel de `.venv` (*Select Kernel*) y usa
   *Restart & Run All*. Sin Cursor:
   `uv run --all-packages --extra kafka --with jupyterlab jupyter lab notebooks/06_tiempo_real.ipynb`
   (`jupyterlab` no es dependencia del proyecto: `--with` lo instala aparte la primera vez,
   con red).

**Si el kernel no tiene `confluent_kafka`**, la celda de parámetros se para con el mensaje
«Falta el extra kafka: uv sync --all-packages --extra kafka»: ejecuta ese comando, reinicia el
kernel y vuelve a empezar. O usa el modo rápido, que no lo necesita.

- **Duración:** con Kafka, unos 30 s: el calentamiento, un día simulado a ×3.600 (24 s
  reales) y la relectura; en modo rápido, unos 5 s.
- **Modo rápido:** con `EDSIM_NB_RAPIDO=1` no hace falta Docker ni el extra. El directo corre
  **en memoria y a velocidad máxima**, y el consumidor es el mismo estado alimentado
  directamente. Es lo que hace el CI (`EDSIM_NB_RAPIDO=1 uv run pytest --nbmake notebooks/`).
  En ese modo **no hay particiones ni latencias reales**: las secciones 5 y 6 lo dicen.
  **Las cifras que cita el texto son las de la ejecución con Kafka.**
- **Las gráficas desaparecen tras cada commit:** el hook `nbstripout` borra los resultados.
- **Mientras corre**, abre Redpanda Console en `http://localhost:8080`: en *Topics* verás
  llegar los mensajes, y en *Consumer Groups*, el grupo de esta lectura.
- Se ejecuta desde la carpeta `notebooks/`: el directo se lanza en la raíz del proyecto.

In [ ]:
import os
import subprocess
import sys
import time
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from edsim.arrivals.profile import load_profile
from edsim.engine.center import load_center
from edsim.engine.engine import simulate_ed
from edsim.engine.events import MemorySink
from edsim.engine.report import MUTED_COLOR, OBSERVED_COLOR, THEORY_COLOR
from edsim.live.census import census, live_events
from edsim.live.clock import SystemClock
from edsim.live.control import MAX_SPEED, NoControl
from edsim.live.kafka import MissingKafkaExtraError, import_confluent_kafka, read_run
from edsim.live.launch import prepare_live, run_prepared
from edsim.live.tracker import LOCATIONS, LiveTracker, ReceivedEvent, fold_states
from edsim.profiles.config import load_profiles

INICIO = time.perf_counter()

# --- Parámetros ------------------------------------------------------------------------
# Con EDSIM_NB_RAPIDO=1 (el CI) el directo corre en memoria: sin Docker ni el extra kafka.
RAPIDO = os.environ.get("EDSIM_NB_RAPIDO") == "1"

RAIZ = Path("..").resolve()  # el notebook se ejecuta desde notebooks/
CONFIG = RAIZ / "config"
KAFKA = "localhost:9092"
SEMILLA = 2026
DIAS, CAL = 3, 2  # tres días: dos de calentamiento y uno de directo
W = CAL * 1440  # el inicio del directo, en minutos simulados
HORIZONTE = DIAS * 1440
VELOCIDAD = 3600  # un minuto simulado por cada 1/60 s real: el día de directo en 24 s
RUN_ID = f"nb06_{datetime.now():%Y%m%dT%H%M%S}"  # propio: no se mezcla con otros directos
EDSIM = Path(sys.executable).with_name("edsim")  # la CLI del mismo entorno que el kernel

if not RAPIDO:
    try:
        import_confluent_kafka()
    except MissingKafkaExtraError as error:
        raise RuntimeError(
            f"{error}, y reinicia el kernel; o ejecuta el notebook en modo rápido "
            "(EDSIM_NB_RAPIDO=1)"
        ) from None

# Las siete ubicaciones de LocationState, en el orden del recorrido de un paciente.
NOMBRE = {
    "waiting_triage": "espera triaje",
    "triage": "triaje",
    "waiting_box": "espera box",
    "waiting_doctor": "espera médico",
    "assessment": "valoración",
    "waiting_tests": "espera pruebas",
    "reassessment": "reevaluación",
}
# Los huecos 1 a 7 de la paleta categórica de engine/report.py, en orden fijo.
COLOR = dict(
    zip(
        LOCATIONS,
        ("#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6"),
        strict=True,
    )
)
plt.rcParams["figure.figsize"] = (10, 4)
print("Modo rápido: directo en memoria" if RAPIDO else f"Con Kafka en {KAFKA}: run_id {RUN_ID}")

## 2. Qué son un consumidor y un grupo

Un **consumidor** es un programa que lee mensajes de uno o varios topics. Kafka no se los
envía: el consumidor los **pide** con `poll()`, a su ritmo, y Kafka no los borra al leerlos.
Por eso cada lector puede ir por su sitio: Redpanda Console, este notebook y, en el hito 7,
el consumidor que guardará los eventos en Postgres.

Los consumidores con el mismo `group.id` forman un **grupo de consumidores** y se reparten
las particiones: con 3 particiones y 3 consumidores del mismo grupo, cada uno lee una. El
grupo guarda en el propio Kafka por qué **offset** va en cada partición («confirma» su
posición). Si el consumidor se para y vuelve, sigue donde lo dejó.

Dos decisiones del consumidor del paquete (`read_run`):

- **Un grupo nuevo por lectura** (`edsim-lector-<run_id>-<8 cifras hex>`). Un grupo
  recuerda su posición: una segunda lectura con el mismo grupo empezaría donde acabó la
  primera y no leería nada. Con un grupo nuevo se vuelve a leer todo, y Redpanda Console lo
  muestra aparte, con su **retraso** (*lag*: mensajes del topic que el grupo aún no ha
  confirmado).
- **`auto.offset.reset=earliest`**. Un grupo nuevo no tiene posición guardada, y esta opción
  dice por dónde empezar: por el final (`latest`, lo que hace librdkafka por defecto, que solo
  ve lo que llegue a partir de ahora) o por el principio (`earliest`). Un consumidor de censo
  quiere el principio: el censo se publica de golpe al empezar el directo, quizá antes de que
  el grupo termine de formarse.

Los topics mezclan todos los directos, así que el consumidor filtra por el `run_id` del
**valor** de cada evento. La clave no sirve para eso: la clave de un evento es su
`episode_id`.

La celda siguiente lanza el directo en otro proceso, como harías tú en otra terminal, y lo
lee a la vez. Cada 5 s imprime cómo va.

In [ ]:
def progreso(tracker):
    ultimo = tracker.events[-1].sim_time if tracker.distinct else float("nan")
    print(
        f"  {tracker.distinct} eventos, {len(tracker.open_episodes())} episodios abiertos,"
        f" minuto {ultimo:.0f}"
    )


t0 = time.perf_counter()
tracker = LiveTracker(RUN_ID)
if RAPIDO:
    # El mismo directo, en memoria y sin esperas: el LiveTracker es el sumidero.
    preparado = prepare_live(
        load_center(CONFIG / "centro.toml"),
        load_profile(CONFIG / "llegadas.toml"),
        DIAS,
        SEMILLA,
        warmup_days=CAL,
        run_id=RUN_ID,
        profiles=load_profiles(CONFIG / "perfiles.toml"),
    )
    run_prepared(preparado, clock=SystemClock(), speed=MAX_SPEED, control=NoControl(), sink=tracker)
    lectura = None
else:
    directo = subprocess.Popen(
        [str(EDSIM), "directo", "lanzar", "--dias", str(DIAS), "--calentamiento", str(CAL),
         "--velocidad", str(VELOCIDAD), "--run-id", RUN_ID, "--kafka", KAFKA],
        cwd=RAIZ,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )  # fmt: skip
    lectura = None
    try:
        lectura = read_run(KAFKA, RUN_ID, tracker, on_progress=progreso)
    finally:
        # Si la lectura falla, se para el directo: no queda un proceso huérfano publicando.
        if lectura is None and directo.poll() is None:
            directo.terminate()
            directo.wait(timeout=10)
    salida, errores = directo.communicate(timeout=60)
    print("\nÚltimas líneas del directo:")
    print("\n".join(salida.splitlines()[-3:]))
    if errores.strip():
        print(errores)
    print(f"\nGrupo de esta lectura: {lectura.group_id}")
    print(f"Mensajes leídos (de cualquier directo): {lectura.messages}")
inicio, fin = tracker.run_start, tracker.run_end
print(
    f"Leído en {time.perf_counter() - t0:.1f} s: inicio con un censo de {inicio.census_episodes}"
    f" episodios y {inicio.census_events} eventos; fin con motivo {fin.reason} y"
    f" {fin.published} publicados; recibidos {tracker.distinct} distintos;"
    f" completo: {tracker.complete}"
)

**Para experimentar**

- Con Kafka, mira en Redpanda Console (*Consumer Groups*) el grupo que imprime la celda: al
  terminar, su retraso es 0 en las cinco particiones que lee.
- Cambia `VELOCIDAD` a 600 y vuelve a ejecutar: el día de directo tarda 2 min 24 s, y el
  progreso avanza al ritmo del pacer.

> **Lo que debes recordar:** un consumidor pide los mensajes y Kafka no los borra al leerlos;
> un grupo reparte las particiones entre sus consumidores y guarda su posición, y un grupo
> nuevo con `earliest` lee desde el principio.

## 3. El censo en vivo

El consumidor no recibe una «foto» del servicio: recibe **eventos** y deriva el estado
aplicándolos uno a uno (*event sourcing*, decisión 58). Para cada episodio guarda un
`LocationState`, el mismo reductor que usa la página Replay de la app: cada evento cambia
unas pocas marcas (triaje empezado, box, valoración en curso, pruebas pendientes, sala de
resultados…) y de esas marcas sale la **ubicación**.

El directo empieza con el **censo**: los eventos del calentamiento de los episodios que
siguen dentro en W. Por eso, nada más leerlo, el consumidor ya tiene el servicio lleno, y
después solo aplica los eventos en vivo. La tabla da la ocupación en W (el censo) y al
final; la gráfica, la ocupación al final de cada hora simulada del directo.

In [ ]:
def fila(etiqueta, ocupacion):
    celdas = "".join(f"{ocupacion.by_location[u]:>15}" for u in LOCATIONS)
    return f"{etiqueta:<16}{celdas}{ocupacion.open:>10}{ocupacion.results_room:>8}"


print(f"{'':<16}" + "".join(f"{NOMBRE[u]:>15}" for u in LOCATIONS) + f"{'abiertos':>10}{'sala':>8}")
print(fila(f"en W ({W} min)", tracker.occupancy_at(W)))
print(fila("al final", tracker.occupancy()))
print(f"\nPlantilla vigente: {tracker.staff}")

horas = np.arange(W, HORIZONTE + 1, 60)
ocupacion = [tracker.occupancy_at(t) for t in horas]
fig, ax = plt.subplots()
ax.stackplot(
    (horas - W) / 60,
    [[o.by_location[u] for o in ocupacion] for u in LOCATIONS],
    labels=[NOMBRE[u] for u in LOCATIONS],
    colors=[COLOR[u] for u in LOCATIONS],
    edgecolor="white",
    linewidth=0.8,
)
ax.set_xlabel("horas desde el inicio del directo (W)")
ax.set_ylabel("episodios abiertos")
ax.set_title("Ocupación por ubicación, leída en vivo")
ax.set_xlim(0, (HORIZONTE - W) / 60)
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False)
ax.grid(axis="y", color="#e4e3df", linewidth=0.6)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

En las ejecuciones con Kafka del 9 de noviembre, el censo en W son 113 episodios abiertos y
520 eventos, como en la pestaña del hito: 56 esperan box y 25, triaje. Al final del día de
directo quedan 81 abiertos, 7 de ellos en la sala de resultados.

**Para experimentar**

- Cambia `horas` a un paso de 15 minutos: la ocupación tiene más ruido a esa escala.
- Busca la hora con más pacientes esperando médico (`by_location["waiting_doctor"]`). ¿A qué
  hora del día cae?

> **Lo que debes recordar:** el consumidor deriva el estado aplicando eventos, con el mismo
> código para el censo y para el directo; el censo le da el servicio lleno desde el primer
> momento.

## 4. Consumidor = batch

La prueba central del hito, vista desde el consumidor: con la misma semilla y el mismo
`run_id`, el **batch** (`simulate_ed`, sin pacer ni Kafka) produce el log completo. Si se
filtra como el directo (el censo en W más los eventos desde W) y se le aplica el mismo
reductor, el estado de **cada** episodio tiene que ser el que el consumidor ha leído en vivo.
Ni el ritmo, ni las particiones, ni los repetidos deben cambiar nada.

In [ ]:
batch = MemorySink()
simulate_ed(
    load_center(CONFIG / "centro.toml"),
    load_profile(CONFIG / "llegadas.toml"),
    DIAS,
    SEMILLA,
    sink=batch,
    warmup_days=CAL,
    run_id=RUN_ID,
    profiles=load_profiles(CONFIG / "perfiles.toml"),
)
filtrado = census(batch.events, W) + live_events(batch.events, W)
referencia = fold_states(filtrado)
distintos = [e for e in referencia if referencia[e] != tracker.states.get(e)]
print(f"Batch: {len(batch.events)} eventos; filtrado como el directo: {len(filtrado)}")
print(f"Episodios: {len(referencia)} en el batch filtrado, {len(tracker.states)} leídos")
print(f"Episodios con un estado distinto: {len(distintos)}")
print(f"Consumidor = batch: {tracker.states == referencia}")

En la ejecución con Kafka: 13.157 eventos en el batch, 5.048 tras filtrar (los mismos que
publica el directo), 456 episodios y 0 con un estado distinto.

> **Lo que debes recordar:** lo que un consumidor reconstruye en vivo es exactamente lo que da
> el batch filtrado con el mismo `run_id`; si no lo fuera, el directo habría perdido,
> duplicado o desordenado algo que importa.

## 5. Particiones y orden

`ed.patient.events` tiene **3 particiones**, y la clave de cada evento es su `episode_id`:
todos los eventos de un paciente van a la misma partición y, dentro de ella, se leen en el
orden en que se publicaron. Entre particiones no hay orden: el consumidor puede recibir un
evento de las 10:05 de la partición 2 después de uno de las 10:20 de la partición 0.

El estado necesita el orden **dentro de cada episodio**, no el global, y por eso se cumple
siempre (`episode_order_errors` debe ser 0). Las **inversiones entre particiones** (eventos
que llegan con un `sim_time` menor que el máximo ya recibido en otra partición) son
normales: un consumidor que necesite un orden global por tiempo tiene que ordenar él (el
hito 8 lo hará con ventanas).

In [ ]:
if RAPIDO:
    print("Modo rápido: el directo en memoria no tiene particiones; todo llega en una sola")
    print("secuencia, en el orden de publicación, y no hay inversiones.")
for (topic, particion), n in sorted(tracker.by_partition.items(), key=str):
    print(f"  {topic or 'memoria'} [{particion if particion is not None else '-'}]: {n} eventos")
print(f"Errores de orden dentro de un episodio: {tracker.episode_order_errors}")
print(f"Inversiones entre particiones: {tracker.cross_partition_inversions}")

En las ejecuciones con Kafka, las tres particiones de `ed.patient.events` recibieron 1.585,
1.529 y 1.914 eventos y `ed.resource.events`, 20; 0 errores de orden dentro de un episodio y
entre 650 y 710 inversiones entre particiones (cambia en cada ejecución: depende de cómo
agrupa Kafka cada entrega).

**Para experimentar**

- Con Kafka, en Redpanda Console, abre `ed.patient.events` y filtra por la clave de un
  episodio: todos sus mensajes están en la misma partición, con offsets crecientes.

> **Lo que debes recordar:** la clave garantiza el orden de un episodio, no el global; el
> consumidor no necesita más para el censo, y quien quiera el orden por tiempo lo reconstruye.

## 6. Latencias

Cada mensaje lleva una **marca de tiempo** que pone el broker al guardarlo
(`LogAppendTime`, decisión 56): es la hora de llegada a Kafka. La **latencia del broker al
consumidor** es la hora a la que el consumidor recibe el mensaje menos esa marca. Incluye lo
que el consumidor tarda en pedirlo (`poll` cada 0,1 s) y lo que Kafka agrupa antes de
entregar. No es la latencia de extremo a extremo: la espera del productor (`linger.ms`, unos
5 ms) va antes de la marca.

No se compara con `event_time`: esa es la hora **simulada** del evento. A ×3.600 el reloj
simulado corre una hora por segundo real, y la diferencia no mide nada de la tubería.

In [ ]:
resumen = tracker.latency_summary()
if resumen is None:
    print("Modo rápido: sin broker no hay marcas de llegada, y no hay latencias que medir.")
else:
    print(
        f"{resumen.count} eventos: mediana {resumen.median_ms:.1f} ms,"
        f" p95 {resumen.p95_ms:.1f} ms, máximo {resumen.max_ms:.1f} ms"
    )
    fig, ax = plt.subplots(figsize=(9, 3.5))
    ax.hist(tracker.latencies_ms, bins=60, color=OBSERVED_COLOR, edgecolor="white", linewidth=0.5)
    # La mediana se rotula a la izquierda de su línea y el p95 a la derecha: no se pisan.
    for valor, nombre, color, lado in (
        (resumen.median_ms, "mediana", MUTED_COLOR, -1),
        (resumen.p95_ms, "p95", THEORY_COLOR, 1),
    ):
        ax.axvline(valor, color=color, linewidth=2)
        ax.annotate(
            f"{nombre} {valor:.1f} ms",
            (valor, ax.get_ylim()[1] * 0.9),
            xytext=(5 * lado, 0),
            textcoords="offset points",
            ha="left" if lado > 0 else "right",
            color="#3d3c38",
        )
    ax.set_xlabel("del broker al consumidor (ms)")
    ax.set_ylabel("eventos")
    ax.set_title("Latencia de cada evento leído")
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    plt.show()

En dos ejecuciones con Kafka: mediana de 1,6 y 1,7 ms, p95 de 4,9 y 3,6 ms y máximo de 10,8
y 35,1 ms. Casi todo es el `poll` del consumidor; los picos raros son pausas del sistema.

> **Lo que debes recordar:** la hora de llegada la pone el broker (`LogAppendTime`); la
> latencia del consumidor se mide contra ella, nunca contra la hora simulada del evento.

## 7. Idempotencia

De extremo a extremo, la garantía de EDsim es **«al menos una vez»** (decisión 55): el
productor no duplica (es idempotente), pero un consumidor que se cae después de procesar un
mensaje y antes de confirmar su offset lo vuelve a leer al volver. Por eso el consumidor
también es **idempotente**: si ya ha visto un `event_id`, lo cuenta como repetido y no
cambia nada.

La celda simula el peor caso: **todo** el directo llega dos veces. Con Kafka, una segunda
lectura con otro grupo lo vuelve a leer entero, y sus eventos se pasan al mismo estado; en
modo rápido, se vuelven a pasar los mismos eventos.

In [ ]:
antes = dict(tracker.states)
if RAPIDO:
    segunda = tracker.events
else:
    otra = LiveTracker(RUN_ID)
    relectura = read_run(KAFKA, RUN_ID, otra)
    print(f"Segunda lectura, grupo {relectura.group_id}: {otra.distinct} eventos, completa:"
          f" {relectura.complete}")  # fmt: skip
    segunda = otra.events
for evento in segunda:
    tracker.add(ReceivedEvent(evento))
print(f"Repetidos: {tracker.duplicates}; distintos: {tracker.distinct}")
print(f"El estado no cambia: {tracker.states == antes}")

> **Lo que debes recordar:** «al menos una vez» quiere decir que un mensaje puede llegar
> repetido; el `event_id` determinista (decisión 27) deja descartarlo, y entonces leer dos
> veces es inofensivo.

## 8. Cinco ideas del hito, explicadas

**Por qué el pacer espera hasta instantes absolutos.** Cada `sleep` se pasa un poco (décimas
de milisegundo), y si se durmiera «la diferencia con el evento anterior» esos excesos se
sumarían: 3,8 s de deriva en un día a ×60. El pacer calcula para cada evento su instante
real a partir de un **ancla** (w₀, s₀) y de la velocidad: w = w₀ + (t − s₀)·60/v, y espera
hasta él. Un retraso no se arrastra, porque el siguiente objetivo no depende de cuándo salió
el anterior. Al cambiar la velocidad o reanudar, se reancla en el minuto simulado de ese
momento: el reloj simulado ni salta ni retrocede.

**Qué garantiza y qué no garantiza una clave de Kafka.** La clave decide la partición
(murmur2 de la clave, módulo 3), y dentro de una partición el orden es el de publicación:
todos los eventos de un episodio llegan juntos y en orden. **No** garantiza el orden entre
episodios de particiones distintas, ni entre topics (el `fin` de `ed.runs` puede leerse
antes que los últimos eventos), ni que un mensaje llegue una sola vez. Y solo funciona si
todos los productores usan el mismo hash: por eso EDsim configura `murmur2_random`, el de
Java.

**Por qué el productor es idempotente y el consumidor también.** El productor reintenta
cuando no recibe la confirmación; sin idempotencia, si el broker sí había guardado el
mensaje, el reintento lo duplicaría o lo desordenaría. Con `enable.idempotence` el broker
numera los mensajes de cada productor y descarta los repetidos. Pero eso no cubre al
consumidor: si se cae entre procesar un mensaje y confirmar su offset, lo vuelve a leer. Por
eso el consumidor descarta por `event_id`: las dos mitades juntas dan el efecto de «una
sola vez».

**Por qué `ingest_time` no va dentro del mensaje.** `ingest_time` es la hora a la que el
evento llega al broker, y el productor no la sabe: solo sabe cuándo lo envía. Si la pusiera
él, sería una hora inventada, y el valor dejaría de ser la misma línea del JSONL del batch.
El broker ya guarda esa hora en la marca `LogAppendTime` de cada mensaje; el consumidor del
hito 7 la copiará a la capa bronce.

**Por qué Kafka necesita dos listeners en Docker.** Un cliente se conecta en dos pasos:
llama a la dirección de arranque y el broker le contesta con la dirección a la que debe
volver a llamar (la **anunciada**). Desde WSL2, el simulador llega por `localhost:9092`;
desde dentro de Docker, Redpanda Console llega por `kafka:19092`, porque `localhost` dentro
de un contenedor es el propio contenedor. Un solo listener anunciaría una dirección que uno
de los dos lados no alcanza. Por eso hay un listener `EXTERNAL` y otro `INTERNAL`.

## 9. Resumen

2. Un consumidor pide los mensajes y no los borra; un grupo nuevo con `earliest` lee desde
   el principio, y cada lectura usa su grupo.
3. El censo en vivo se deriva aplicando eventos con `LocationState`, el mismo reductor para
   el censo y el directo.
4. Lo leído en vivo es el batch filtrado con el mismo `run_id`, episodio a episodio.
5. La clave ordena cada episodio; entre particiones no hay orden, y no hace falta.
6. La latencia se mide contra la marca `LogAppendTime` del broker.
7. Con «al menos una vez», el consumidor descarta los repetidos por `event_id`.

In [ ]:
print(f"Tiempo total: {time.perf_counter() - INICIO:.0f} s")